# LeetCode 260 — Single Number III bằng Python

Link đề: https://leetcode.com/problems/single-number-iii/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Array, XOR, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def singleNumber(self, nums: List[int]) -> List[int]:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ file đề có sẵn (bài này không phải Premium). Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (đếm số lần xuất hiện bằng `Counter`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng có đúng **hai** phần tử xuất hiện **một lần**, mọi phần tử khác xuất hiện **đúng hai lần**; tìm hai phần tử đơn lẻ đó với **thời gian tuyến tính** và **bộ nhớ phụ hằng số**.
2. Nắm phép **XOR (`^`)** và ba tính chất then chốt: `a ^ a = 0`, `a ^ 0 = a`, giao hoán/kết hợp — nên XOR cả mảng **triệt tiêu mọi cặp**.
3. Hiểu cách **tách hai số khác nhau thành hai nhóm** bằng **một bit mà hai số khác nhau**, tìm bit đó bằng **`x & -x`** (lấy bit `1` thấp nhất).
4. Đi từ **đếm bằng hai vòng lặp** `O(n²)` → **bảng đếm (`Counter`)** → **sắp xếp** → **tập bật/tắt (set toggle)** → **XOR + phân nhóm theo bit** `O(n)` thời gian, `O(1)` bộ nhớ.
5. Tránh các lỗi kinh điển: trả về `XOR` của hai số, giả định bit `0` luôn khác nhau, nhầm `x & (x - 1)` với `x & -x`, quên khởi tạo `a = 0`, và hiểu nhầm cách `&` hoạt động với số **âm**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`, trong đó **đúng hai phần tử xuất hiện một lần** và **mọi phần tử còn lại xuất hiện đúng hai lần**. Tìm hai phần tử chỉ xuất hiện một lần. Có thể trả về theo **thứ tự bất kỳ**.

Bạn phải viết thuật toán chạy trong **thời gian tuyến tính** và chỉ dùng **bộ nhớ phụ hằng số**.

**Ví dụ 1**

- Input: `nums = [1,2,1,3,2,5]`
- Output: `[3,5]`
- Giải thích: `[5, 3]` cũng là đáp án hợp lệ.

**Ví dụ 2**

- Input: `nums = [-1,0]`
- Output: `[-1,0]`

**Ví dụ 3**

- Input: `nums = [0,1]`
- Output: `[1,0]`

**Ràng buộc:** `2 <= nums.length <= 3 * 10^4`; `-2^31 <= nums[i] <= 2^31 - 1`; mỗi số nguyên trong `nums` xuất hiện **hai lần**, chỉ **hai số nguyên** xuất hiện một lần.

In [1]:
from typing import List
from collections import Counter
import random
import time

examples = [
    ([1, 2, 1, 3, 2, 5], [3, 5]),
    ([-1, 0], [-1, 0]),
    ([0, 1], [1, 0]),
]


def normalize(pair):
    return sorted(pair)


for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {normalize(expected)} (thứ tự bất kỳ)")

nums=[1, 2, 1, 3, 2, 5] -> kỳ vọng [3, 5] (thứ tự bất kỳ)
nums=[-1, 0] -> kỳ vọng [-1, 0] (thứ tự bất kỳ)
nums=[0, 1] -> kỳ vọng [0, 1] (thứ tự bất kỳ)


## 3. Kiến thức nền

### 3.1. Số nhị phân và toán tử bit trong Python

Mỗi số nguyên là dãy bit. `bin(5) = '0b101'`. Toán tử: `&` (AND), `|` (OR), `^` (XOR), `<<`, `>>` (dịch bit). Để quan sát, in dạng nhị phân cố định bằng `format(x, "04b")`.
#### Code demo

In [2]:
a, b = 5, 3
print(f"{a:04b} = {a}")      # 0101
print(f"{b:04b} = {b}")      # 0011
print(f"{a & b:04b} = {a & b}   (AND: 1 nếu cả hai bit đều 1)")
print(f"{a | b:04b} = {a | b}   (OR : 1 nếu ít nhất một bit là 1)")
print(f"{a ^ b:04b} = {a ^ b}   (XOR: 1 nếu hai bit KHÁC nhau)")

0101 = 5
0011 = 3
0001 = 1   (AND: 1 nếu cả hai bit đều 1)
0111 = 7   (OR : 1 nếu ít nhất một bit là 1)
0110 = 6   (XOR: 1 nếu hai bit KHÁC nhau)


### 3.2. XOR: ba tính chất và hệ quả

- `a ^ a = 0` (hai số giống nhau triệt tiêu);
- `a ^ 0 = a`;
- **giao hoán** và **kết hợp**: thứ tự không quan trọng.

Hệ quả: XOR mọi phần tử của một mảng mà **mỗi số xuất hiện hai lần** cho `0`, bất kể thứ tự. Nếu có **một** số xuất hiện một lần, XOR cả mảng cho chính nó (bài **Single Number**, LeetCode 136).
#### Code demo

In [3]:
from functools import reduce
from operator import xor

print(7 ^ 7, 7 ^ 0)                                       # 0 7
print(reduce(xor, [4, 1, 2, 1, 2]))                       # 4: các cặp triệt tiêu
print(reduce(xor, [1, 2, 1, 3, 2, 5]), "= 3 ^ 5 =", 3 ^ 5)   # còn lại XOR của HAI số đơn lẻ

0 7
4
6 = 3 ^ 5 = 6


### 3.3. Với hai số đơn lẻ `a` và `b`, XOR cả mảng cho `a ^ b`

XOR cả mảng triệt tiêu mọi cặp, còn lại `x = a ^ b`. Vì `a ≠ b` nên `x ≠ 0`. Chỗ nào `x` có bit `1` là chỗ `a` và `b` **khác nhau**. Nhưng `x` không cho biết riêng `a` hay `b`. Ta cần **tách** mảng thành hai nhóm sao cho `a` và `b` rơi vào **hai nhóm khác nhau**.
#### Code demo

In [4]:
a, b = 3, 5
x = a ^ b
print(f"a={a:03b}, b={b:03b}, x=a^b={x:03b} ({x})")
print("các bit 1 của x là những vị trí a và b khác nhau")

a=011, b=101, x=a^b=110 (6)
các bit 1 của x là những vị trí a và b khác nhau


### 3.4. Lấy bit `1` thấp nhất: `x & -x`

Với số nguyên `x ≠ 0`, `x & -x` cho số **chỉ có đúng bit `1` thấp nhất của `x`**. Lý do: `-x` là bù hai: `-x = ~x + 1`; cộng `1` làm các bit `0` thấp lật thành `1` rồi nhớ tràn đến đúng vị trí bit `1` thấp nhất của `x`; từ đó trở xuống `x` và `-x` có cùng bit, còn bit cao hơn thì đối nhau, nên AND chỉ giữ đúng bit thấp nhất. Python dùng số nguyên **không giới hạn** với biểu diễn bù hai "vô hạn" nên `x & -x` cũng đúng với số **âm**.
#### Code demo

In [5]:
for x in (6, 12, 40, 1, -8, -6):
    low = x & -x
    print(f"x={x:>3}: x & -x = {low:>2} (nhị phân {format(low, 'b')})")
x = 0b1100
print(f"{x:04b} & -x = {x & -x:04b}; x & (x-1) = {x & (x - 1):04b}  (chú ý: x & (x-1) là XÓA bit thấp nhất, khác hẳn)")

x=  6: x & -x =  2 (nhị phân 10)
x= 12: x & -x =  4 (nhị phân 100)
x= 40: x & -x =  8 (nhị phân 1000)
x=  1: x & -x =  1 (nhị phân 1)
x= -8: x & -x =  8 (nhị phân 1000)
x= -6: x & -x =  2 (nhị phân 10)
1100 & -x = 0100; x & (x-1) = 1000  (chú ý: x & (x-1) là XÓA bit thấp nhất, khác hẳn)


### 3.5. Phân nhóm theo một bit

Cho bit mặt nạ `lb` (đúng một bit `1`). Mỗi số `v` thuộc **nhóm 1** nếu `v & lb != 0`, ngược lại **nhóm 0**. Nếu `lb` là một bit mà `a` và `b` khác nhau thì `a` và `b` thuộc **hai nhóm khác nhau**. Mỗi **cặp trùng** `(v, v)` vì giống nhau nên **cùng nhóm**. Do đó **trong mỗi nhóm**: các cặp triệt tiêu khi XOR, chỉ còn đúng một số đơn lẻ.
#### Code demo

In [6]:
nums = [1, 2, 1, 3, 2, 5]
lb = (3 ^ 5) & -(3 ^ 5)
print("lb =", lb, f"({lb:03b})")
for v in nums:
    print(f"v={v} ({v:03b}) & lb={lb:03b} -> {'nhóm 1' if v & lb else 'nhóm 0'}")

lb = 2 (010)
v=1 (001) & lb=010 -> nhóm 0
v=2 (010) & lb=010 -> nhóm 1
v=1 (001) & lb=010 -> nhóm 0
v=3 (011) & lb=010 -> nhóm 1
v=2 (010) & lb=010 -> nhóm 1
v=5 (101) & lb=010 -> nhóm 0


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1 `[1,2,1,3,2,5]` (hai số đơn lẻ là `3` và `5`):

1. XOR cả mảng: `1^2^1^3^2^5 = (1^1)^(2^2)^3^5 = 3 ^ 5 = 6 = 110`.
2. Chọn một bit mà `3 = 011` và `5 = 101` khác nhau: bit `1` của `x = 110` (giá trị `010 = 2`) — bit thấp nhất của `x`.
3. Chia theo bit `010`:
   - có bit `010`: `2` (`010`), `2`, `3` (`011`) → XOR `= 2 ^ 2 ^ 3 = 3`;
   - không có bit `010`: `1` (`001`), `1`, `5` (`101`) → XOR `= 1 ^ 1 ^ 5 = 5`.
4. Hai số đơn lẻ là `3` và `5`.

**Cách nghĩ:** biến bài toán "hai số lẻ loi" thành **hai** bài toán "một số lẻ loi" (đã biết giải bằng XOR) nhờ một phép chia nhóm khéo léo.

## 5. Cách 1 — Brute Force / baseline: đếm từng phần tử bằng hai vòng lặp

**Ý tưởng.** Với mỗi `nums[i]`, đếm số lần nó xuất hiện trong mảng (một vòng lặp lồng); nếu đếm được `1` thì thu vào kết quả.
#### Code demo

In [7]:
def single_number_bruteforce(nums):
    result = []
    for i, x in enumerate(nums):
        occurrences = 0
        for y in nums:
            if y == x:
                occurrences += 1
        if occurrences == 1:
            result.append(x)
    return result


for nums, expected in examples:
    print(nums, "->", single_number_bruteforce(nums), "(kỳ vọng", normalize(expected), "thứ tự bất kỳ)")

[1, 2, 1, 3, 2, 5] -> [3, 5] (kỳ vọng [3, 5] thứ tự bất kỳ)
[-1, 0] -> [-1, 0] (kỳ vọng [-1, 0] thứ tự bất kỳ)
[0, 1] -> [0, 1] (kỳ vọng [0, 1] thứ tự bất kỳ)


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`: **Time `O(n²)`**; với `n = 3·10^4` là `~9·10^8` phép — **quá chậm**; **Space `O(1)`**.

**Nút thắt.** Đếm từng phần tử bằng cách quét lại cả mảng. Ta có thể đếm **tất cả** trong một lượt bằng bảng băm — nhưng tốn `O(n)` bộ nhớ; đề muốn `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**XOR cả mảng** cho `x = a ^ b` (hai số đơn lẻ), `x ≠ 0`.

**Tìm một bit phân biệt:** `lb = x & -x` (bit `1` thấp nhất của `x`), là một vị trí mà `a` và `b` khác nhau.

**Chia nhóm và XOR trong từng nhóm:** các số có bit `lb` bật vào nhóm 1, số còn lại vào nhóm 0. `a` và `b` rơi vào hai nhóm khác nhau; mỗi cặp trùng cùng nhóm nên triệt tiêu. Do đó XOR nhóm 1 ra một số đơn lẻ, và số kia là `x ^ (số vừa tìm)`.

Chỉ cần **hai lượt duyệt**, vài biến số nguyên: **`O(n)` thời gian, `O(1)` bộ nhớ**.

## 7. Các cách tiếp theo

### Cách 2 — Bảng đếm (`Counter` / dict)

**Giải thích.** Đếm số lần xuất hiện của mọi phần tử bằng `Counter`, rồi lấy các phần tử có số lần bằng `1`.
#### Code demo

In [8]:
def single_number_counter(nums):
    counts = Counter(nums)
    return [value for value, c in counts.items() if c == 1]


for nums, expected in examples:
    print(nums, "->", single_number_counter(nums), "(kỳ vọng", normalize(expected), ")")

[1, 2, 1, 3, 2, 5] -> [3, 5] (kỳ vọng [3, 5] )
[-1, 0] -> [-1, 0] (kỳ vọng [-1, 0] )
[0, 1] -> [0, 1] (kỳ vọng [0, 1] )


**Độ phức tạp Cách 2:** **Time `O(n)`** (trung bình), **Space `O(n)`** — không thỏa yêu cầu `O(1)` nhưng rất thực dụng.

### Cách 3 — Sắp xếp rồi quét theo cặp

**Giải thích.** Sắp xếp; các số bằng nhau đứng cạnh nhau. Duyệt `i = 0, 2, 4, ...`: nếu `nums[i] != nums[i+1]` thì `nums[i]` là số đơn lẻ (rồi dịch `i` lên `1` thay vì `2`).
#### Code demo

In [9]:
def single_number_sort(nums):
    a = sorted(nums)
    result = []
    i = 0
    while i < len(a):
        if i + 1 < len(a) and a[i] == a[i + 1]:
            i += 2                                   # một cặp trùng
        else:
            result.append(a[i])                      # số đơn lẻ
            i += 1
    return result


for nums, expected in examples:
    print(nums, "->", single_number_sort(nums), "(kỳ vọng", normalize(expected), ")")

[1, 2, 1, 3, 2, 5] -> [3, 5] (kỳ vọng [3, 5] )
[-1, 0] -> [-1, 0] (kỳ vọng [-1, 0] )
[0, 1] -> [0, 1] (kỳ vọng [0, 1] )


**Độ phức tạp Cách 3:** **Time `O(n log n)`**, **Space `O(n)`** (bản sao đã sắp; `O(1)` nếu sắp tại chỗ).

### Cách 4 — Tập bật/tắt (set toggle)

**Giải thích.** Duyệt từng số: nếu đã có trong tập thì **bỏ ra**, chưa có thì **thêm vào**. Cuối cùng trong tập còn đúng hai số xuất hiện lẻ lần (một lần).
#### Code demo

In [10]:
def single_number_set_toggle(nums):
    seen = set()
    for x in nums:
        if x in seen:
            seen.remove(x)
        else:
            seen.add(x)
    return list(seen)


for nums, expected in examples:
    print(nums, "->", normalize(single_number_set_toggle(nums)), "(kỳ vọng", normalize(expected), ")")

[1, 2, 1, 3, 2, 5] -> [3, 5] (kỳ vọng [3, 5] )
[-1, 0] -> [-1, 0] (kỳ vọng [-1, 0] )
[0, 1] -> [0, 1] (kỳ vọng [0, 1] )


**Độ phức tạp Cách 4:** **Time `O(n)`** (trung bình), **Space `O(n)`** (tập có thể chứa tới `n/2 + 1` phần tử).

### Cách 5 — XOR + phân nhóm theo bit thấp nhất (cách nộp chuẩn)

**Giải thích.** Như mục 6: `xs = XOR cả mảng`; `lb = xs & -xs`; `a = XOR các số có bit lb`; `b = xs ^ a`.
#### Code demo

In [11]:
def single_number_xor(nums):
    xs = 0
    for x in nums:
        xs ^= x                                      # xs = a ^ b
    lb = xs & -xs                                    # bit 1 thấp nhất của xs: a và b khác nhau ở bit này
    a = 0
    for x in nums:
        if x & lb:                                   # nhóm có bit lb bật
            a ^= x                                   # cặp trùng triệt tiêu, còn đúng một số đơn lẻ
    b = xs ^ a
    return [a, b]


for nums, expected in examples:
    print(nums, "->", single_number_xor(nums), "(kỳ vọng", normalize(expected), "thứ tự bất kỳ)")

[1, 2, 1, 3, 2, 5] -> [3, 5] (kỳ vọng [3, 5] thứ tự bất kỳ)
[-1, 0] -> [-1, 0] (kỳ vọng [-1, 0] thứ tự bất kỳ)
[0, 1] -> [1, 0] (kỳ vọng [0, 1] thứ tự bất kỳ)


In [12]:
# Code nộp LeetCode cho Cách 5
from functools import reduce
from operator import xor


class Solution:
    def singleNumber(self, nums: List[int]) -> List[int]:
        xs = reduce(xor, nums)
        a = 0
        lb = xs & -xs
        for x in nums:
            if x & lb:
                a ^= x
        b = xs ^ a
        return [a, b]


print(Solution().singleNumber([1, 2, 1, 3, 2, 5]))

[3, 5]


**Độ phức tạp Cách 5:** **Time `O(n)`** (hai lượt duyệt), **Space `O(1)`** (vài biến số nguyên).

> Cách 5 là bản nộp tối ưu và là câu trả lời mong đợi cho yêu cầu "tuyến tính + hằng số". Cách 2 và 4 là cách thực dụng khi không bị ràng buộc bộ nhớ.

## 8. TRACE từng bước

Ta in từng bước của Cách 5 trên `[1, 2, 1, 3, 2, 5]` và trên `[-1, 0]` (có số âm) với biểu diễn nhị phân.
#### Code demo

In [13]:
def trace_xor(nums, width=4):
    def fmt(v):
        return format(v & ((1 << width) - 1), f"0{width}b")        # chỉ để hiển thị width bit thấp

    print("nums =", nums)
    xs = 0
    for x in nums:
        xs ^= x
        print(f"  xs ^= {x:>3} ({fmt(x)}) -> xs = {xs:>3} ({fmt(xs)})")
    lb = xs & -xs
    print(f"xs = {xs} ({fmt(xs)}), lb = xs & -xs = {lb} ({fmt(lb)})")
    a = 0
    for x in nums:
        in_group = bool(x & lb)
        if in_group:
            a ^= x
        print(f"  x={x:>3} ({fmt(x)}) & lb: {'nhóm 1 -> a ^= x, a = ' + str(a) if in_group else 'nhóm 0 (bỏ qua)'}")
    b = xs ^ a
    print(f"a = {a}, b = xs ^ a = {b}")
    return [a, b]


trace_xor([1, 2, 1, 3, 2, 5])
print()
trace_xor([-1, 0])

nums = [1, 2, 1, 3, 2, 5]
  xs ^=   1 (0001) -> xs =   1 (0001)
  xs ^=   2 (0010) -> xs =   3 (0011)
  xs ^=   1 (0001) -> xs =   2 (0010)
  xs ^=   3 (0011) -> xs =   1 (0001)
  xs ^=   2 (0010) -> xs =   3 (0011)
  xs ^=   5 (0101) -> xs =   6 (0110)
xs = 6 (0110), lb = xs & -xs = 2 (0010)
  x=  1 (0001) & lb: nhóm 0 (bỏ qua)
  x=  2 (0010) & lb: nhóm 1 -> a ^= x, a = 2
  x=  1 (0001) & lb: nhóm 0 (bỏ qua)
  x=  3 (0011) & lb: nhóm 1 -> a ^= x, a = 1
  x=  2 (0010) & lb: nhóm 1 -> a ^= x, a = 3
  x=  5 (0101) & lb: nhóm 0 (bỏ qua)
a = 3, b = xs ^ a = 5

nums = [-1, 0]
  xs ^=  -1 (1111) -> xs =  -1 (1111)
  xs ^=   0 (0000) -> xs =  -1 (1111)
xs = -1 (1111), lb = xs & -xs = 1 (0001)
  x= -1 (1111) & lb: nhóm 1 -> a ^= x, a = -1
  x=  0 (0000) & lb: nhóm 0 (bỏ qua)
a = -1, b = xs ^ a = 0


[-1, 0]

Bảng tóm tắt (`[1, 2, 1, 3, 2, 5]`):

| `x` | nhị phân | `x & lb` (`lb = 010`) | nhóm | `a` sau |
|---|---|---|---|---|
| 1 | `001` | `000` | 0 | 0 |
| 2 | `010` | `010` | 1 | 2 |
| 1 | `001` | `000` | 0 | 2 |
| 3 | `011` | `010` | 1 | 1 (`2 ^ 3`) |
| 2 | `010` | `010` | 1 | 3 (`1 ^ 2`) |
| 5 | `101` | `000` | 0 | 3 |

`a = 3`, `b = xs ^ a = 6 ^ 3 = 5`. Ở nhóm 1 ta có `2, 3, 2` → `3`; ở nhóm 0 ta có `1, 1, 5` → `5`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 5 (XOR + phân nhóm). Đúng `class Solution`, đúng `singleNumber(self, nums: List[int]) -> List[int]`.

In [14]:
class Solution:
    def singleNumber(self, nums: List[int]) -> List[int]:
        xs = 0
        for x in nums:
            xs ^= x                       # xs = a ^ b (các cặp trùng triệt tiêu)
        lb = xs & -xs                     # bit 1 thấp nhất của xs: một bit mà a và b khác nhau
        a = 0
        for x in nums:
            if x & lb:                    # chỉ XOR các số thuộc nhóm có bit lb bật
                a ^= x
        return [a, xs ^ a]


sol = Solution()
for nums, expected in examples:
    assert normalize(sol.singleNumber(nums)) == normalize(expected)
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Triệt tiêu cặp:** XOR giao hoán và kết hợp, `v ^ v = 0`, `0 ^ w = w`; nên XOR cả mảng = XOR của hai số đơn lẻ `a ^ b` (mọi cặp biến mất).
- **`xs ≠ 0`:** vì `a ≠ b` (hai số đơn lẻ khác nhau, nếu bằng nhau chúng sẽ xuất hiện hai lần). Vậy `xs` có ít nhất một bit `1`, và `lb = xs & -xs` là một bit `1` của `xs`.
- **`a` và `b` khác nhau ở bit `lb`:** bit `lb` của `xs = a ^ b` là `1` ⇒ bit `lb` của `a` và `b` khác nhau ⇒ đúng một trong hai thuộc nhóm "bit `lb` bật".
- **Mỗi cặp trùng cùng nhóm:** hai số bằng nhau có cùng bit `lb`. Trong nhóm chứa `a` (giả sử), các cặp trùng triệt tiêu, XOR của nhóm bằng `a` (số đơn lẻ duy nhất của nhóm). Số còn lại `b = xs ^ a`.
- **Số âm:** Python dùng bù hai không giới hạn; `x & -x` và `^`, `&` hành xử đúng như trên số có dấu 32-bit, nên cả số âm lẫn `-2^31` đều được xử lý đúng.
- **Thứ tự kết quả:** đề cho phép bất kỳ; ta trả `[a, b]` (có thể khác thứ tự của ví dụ).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa `3·10^4`).

| Cách | Time | Space |
|---|---|---|
| 1. Hai vòng lặp đếm | `O(n²)` | `O(1)` |
| 2. `Counter` | `O(n)` | `O(n)` |
| 3. Sắp xếp + quét cặp | `O(n log n)` | `O(n)` (hoặc `O(1)` nếu sắp tại chỗ) |
| 4. Set toggle | `O(n)` | `O(n)` |
| 5. XOR + phân nhóm | `O(n)` | `O(1)` |

Cách 5 thỏa đúng yêu cầu đề (tuyến tính + hằng số). Phép toán bit trên số nguyên 32-bit là `O(1)`. Không có khác biệt average/worst: luôn đúng hai lượt duyệt (Cách 2, 4 có "trung bình" vì dùng bảng băm).

## 12. Bộ test

`oracle` là brute force độc lập: dùng `Counter` (không dùng XOR) để lấy các phần tử có tần suất `1`, kiểm tra thêm đúng hai phần tử. Ta sinh đầu vào ngẫu nhiên hợp lệ: `m` giá trị xuất hiện hai lần, hai giá trị xuất hiện một lần, trộn ngẫu nhiên.
#### Code demo

In [15]:
def oracle(nums):
    counts = Counter(nums)
    singles = sorted(v for v, c in counts.items() if c == 1)
    assert len(singles) == 2 and all(c in (1, 2) for c in counts.values())
    return singles


def make_input(pairs, singles, rng):
    nums = []
    for v in pairs:
        nums += [v, v]
    nums += list(singles)
    rng.shuffle(nums)
    return nums


impls = {
    "bruteforce": single_number_bruteforce,
    "counter": single_number_counter,
    "sort": single_number_sort,
    "set_toggle": single_number_set_toggle,
    "xor": single_number_xor,
    "submit": sol.singleNumber,
}

for nums, expected in examples:
    assert oracle(nums) == normalize(expected)
    for key, fn in impls.items():
        assert normalize(fn(list(nums))) == normalize(expected), (key, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [16]:
INT_MIN, INT_MAX = -2 ** 31, 2 ** 31 - 1
edge_cases = [
    ([0, 1], [0, 1], "n = 2 nhỏ nhất"),
    ([-1, 0], [-1, 0], "có số âm"),
    ([INT_MIN, INT_MAX], [INT_MIN, INT_MAX], "hai giá trị biên của đề"),
    ([INT_MIN, INT_MIN, 5, INT_MAX, INT_MAX, 7], [5, 7], "cặp là INT_MIN / INT_MAX"),
    ([2, 2, 3, 4, 3, 5, 4, 6], [5, 6], "hai số lẻ cạnh nhau (bit thấp nhất khác ở bit 0)"),
    ([1, 3, 2, 2], [1, 3], "a ^ b = 2: bit 0 của a, b GIỐNG nhau (cẩn thận lỗi giả định bit 0)"),
    ([4, 6, 7, 7, 9, 9], [4, 6], "a ^ b = 2: bit 0 giống nhau, bit 1 khác nhau"),
    ([0, 0, 1, 2], [1, 2], "có số 0 lặp"),
    ([-2, -2, -3, 8], [-3, 8], "âm và dương trộn lẫn"),
    ([100000, 100000, 1, 2 ** 20], [1, 2 ** 20], "bit cao"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == normalize(expected), (label, oracle(nums), expected)
    for key, fn in impls.items():
        assert normalize(fn(list(nums))) == normalize(expected), (key, label, fn(list(nums)), expected)
    print("PASS", label)

PASS n = 2 nhỏ nhất
PASS có số âm
PASS hai giá trị biên của đề
PASS cặp là INT_MIN / INT_MAX
PASS hai số lẻ cạnh nhau (bit thấp nhất khác ở bit 0)
PASS a ^ b = 2: bit 0 của a, b GIỐNG nhau (cẩn thận lỗi giả định bit 0)
PASS a ^ b = 2: bit 0 giống nhau, bit 1 khác nhau
PASS có số 0 lặp
PASS âm và dương trộn lẫn
PASS bit cao


### 12.2. Tính chất và phản ví dụ

- Kết quả luôn **hai số khác nhau**, và `a ^ b == XOR cả mảng`.
- **Bất biến thứ tự:** xáo trộn `nums` không đổi kết quả.
- **Phản ví dụ** cho lỗi "luôn dùng bit `0`" (`xs & 1`): `[1, 3, 2, 2]` có `1 ^ 3 = 2` nên bit `0` giống nhau.
- **Phản ví dụ** cho lỗi nhầm `x & (x - 1)` với `x & -x`.
#### Code demo

In [17]:
rng = random.Random(260)
for _ in range(500):
    pairs = rng.sample(range(-1000, 1000), rng.randint(0, 20) + 2)
    singles, pairs = pairs[:2], pairs[2:]
    nums = make_input(pairs, singles, rng)
    a, b = sol.singleNumber(nums)
    assert a != b and sorted([a, b]) == sorted(singles)
    assert a ^ b == reduce(xor, nums)
    shuffled = nums[:]
    rng.shuffle(shuffled)
    assert sorted(sol.singleNumber(shuffled)) == sorted(singles)
print("PASS tính chất (hai số khác nhau, a ^ b = XOR mảng, bất biến thứ tự) trên 500 mảng ngẫu nhiên")


def sai_dung_bit_0(nums):
    xs = 0
    for x in nums:
        xs ^= x
    a = 0
    for x in nums:
        if x & 1:                                    # SAI: giả định bit 0 của hai số đơn lẻ khác nhau
            a ^= x
    return [a, xs ^ a]


print("phản ví dụ [1,3,2,2]: đúng", sorted(sol.singleNumber([1, 3, 2, 2])), "| dùng bit 0:", sorted(sai_dung_bit_0([1, 3, 2, 2])))
assert sorted(sai_dung_bit_0([1, 3, 2, 2])) != [1, 3]

PASS tính chất (hai số khác nhau, a ^ b = XOR mảng, bất biến thứ tự) trên 500 mảng ngẫu nhiên
phản ví dụ [1,3,2,2]: đúng [1, 3] | dùng bit 0: [0, 2]


### 12.3. Đối chiếu ngẫu nhiên (có seed) và dữ liệu lớn
#### Code demo

In [18]:
rng = random.Random(4260)
for trial in range(2000):
    m = rng.randint(0, 25)
    lo, hi = rng.choice([(-5, 5), (-100, 100), (INT_MIN, INT_MAX)])
    if hi - lo + 1 < m + 2:
        lo, hi = -100, 100
    pool = set()
    while len(pool) < m + 2:
        pool.add(rng.randint(lo, hi))
    values = list(pool)
    rng.shuffle(values)
    singles, pairs = values[:2], values[2:]
    nums = make_input(pairs, singles, rng)
    truth = oracle(nums)
    for key, fn in impls.items():
        assert normalize(fn(list(nums))) == truth, (key, nums, fn(list(nums)), truth)
print("PASS 2000 mảng ngẫu nhiên (cả miền 32-bit đầy đủ, có số âm): mọi cách khớp oracle")

# n = 3 * 10^4: bỏ vét cạn O(n^2); so các cách còn lại
pool = rng.sample(range(INT_MIN, INT_MAX), 15_000)
singles, pairs = pool[:2], pool[2:2 + 14_999]
big = make_input(pairs, singles, rng)
print("n =", len(big))
truth = sorted(singles)
for key in ("counter", "sort", "set_toggle", "xor", "submit"):
    t0 = time.perf_counter()
    value = normalize(impls[key](list(big)))
    dt = (time.perf_counter() - t0) * 1000
    assert value == truth, key
    print(f"{key:<11} {dt:7.2f} ms")
print("PASS n ~ 3*10^4")

PASS 2000 mảng ngẫu nhiên (cả miền 32-bit đầy đủ, có số âm): mọi cách khớp oracle


n = 29998
counter        2.55 ms


sort           7.81 ms
set_toggle     2.25 ms
xor            3.43 ms
submit         3.11 ms
PASS n ~ 3*10^4


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về XOR của cả mảng (nhầm `a ^ b` là đáp án)
#### Code sai

In [19]:
def sai_tra_xor(nums):
    xs = 0
    for x in nums:
        xs ^= x
    return [xs]                                       # SAI: xs = a ^ b, không phải hai số a, b


print("đúng:", sorted(sol.singleNumber([1, 2, 1, 3, 2, 5])), "| sai:", sai_tra_xor([1, 2, 1, 3, 2, 5]))

đúng: [3, 5] | sai: [6]


**Vì sao sai:** với hai số đơn lẻ, XOR cả mảng chỉ cho `a ^ b` (một con số), mất thông tin riêng từng số. Cần bước **phân nhóm** để tách chúng ra.

### Lỗi 2: Giả định bit `0` (hoặc một bit cố định) luôn phân biệt hai số
Xem `sai_dung_bit_0` ở phần test: `[1, 3, 2, 2]` có `1` và `3` giống nhau ở bit `0` nên cả hai rơi vào cùng nhóm, kết quả sai. **Code đúng:** chọn bit từ chính `xs` (`xs & -xs`), vì đó là bit đảm bảo `a` và `b` khác nhau.

### Lỗi 3: Dùng `xs & (xs - 1)` thay vì `xs & -xs`
#### Code sai

In [20]:
def sai_xoa_bit_thap(nums):
    xs = 0
    for x in nums:
        xs ^= x
    mask = xs & (xs - 1)                               # SAI: đây là XÓA bit 1 thấp nhất, không phải GIỮ lại nó
    a = 0
    for x in nums:
        if x & mask:
            a ^= x
    return [a, xs ^ a]


for nums in ([3, 1, 5, 5], [1, 2, 1, 3, 2, 5]):
    print(nums, "đúng:", sorted(sol.singleNumber(nums)), "| sai:", sorted(sai_xoa_bit_thap(nums)))

[3, 1, 5, 5] đúng: [1, 3] | sai: [0, 2]
[1, 2, 1, 3, 2, 5] đúng: [3, 5] | sai: [3, 5]


**Vì sao sai:** `xs & (xs - 1)` **xóa** bit `1` thấp nhất; khi `xs` chỉ có đúng một bit `1` (như `[3,1,5,5]` với `xs = 2`) kết quả là `0` nên mọi số rơi vào nhóm 0 và ta trả `[0, 2]` (sai). Khi `xs` có nhiều bit `1` thì đôi khi vẫn "may mắn" đúng (ví dụ ca thứ hai). Cần **giữ đúng bit thấp nhất**: `xs & -xs`.

### Lỗi 4: Cộng (`+`) thay vì XOR (`^`) trong nhóm
#### Code sai

In [21]:
def sai_cong_thay_xor(nums):
    xs = 0
    for x in nums:
        xs ^= x
    lb = xs & -xs
    a = 0
    for x in nums:
        if x & lb:
            a += x                                      # SAI: cộng không làm các cặp trùng triệt tiêu
    return [a, xs ^ a]


print("đúng:", sorted(sol.singleNumber([1, 2, 1, 3, 2, 5])), "| sai:", sorted(sai_cong_thay_xor([1, 2, 1, 3, 2, 5])))

đúng: [3, 5] | sai: [1, 7]


**Vì sao sai:** chỉ XOR mới có `v ^ v = 0`; cộng `2 + 3 + 2 = 7` chứ không phải `3`. Phải dùng `a ^= x`.

### Lỗi 5: Dùng bảng đếm / sắp xếp mà tưởng đã thỏa yêu cầu `O(1)`
`Counter` và set toggle tốn `O(n)` bộ nhớ; sắp xếp tốn `O(n log n)` thời gian. Chúng **đúng** nhưng **không đáp ứng** yêu cầu "tuyến tính + hằng số" của đề. Hãy biết nói rõ sự đánh đổi khi nộp hoặc phỏng vấn.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**mọi phần tử xuất hiện hai lần, trừ một (hoặc hai) phần tử**" + yêu cầu `O(1)` bộ nhớ → **XOR** (`a ^ a = 0`).
- Với **một** số đơn lẻ (Single Number I): XOR cả mảng. Với **hai** số (bài này): XOR rồi **phân nhóm bằng một bit phân biệt**.
- **Bit phân biệt:** `lb = xs & -xs` (bit `1` thấp nhất). Nhớ: `x & -x` **giữ** bit thấp nhất; `x & (x - 1)` **xóa** nó.
- Với **ba lần** (Single Number II, LeetCode 137): đếm bit theo vị trí modulo `3`, hoặc dùng máy trạng thái hai bit.
- Python: số nguyên không giới hạn nên `x & -x` đúng cho số âm; trong C++/Java cẩn thận `INT_MIN` (`-x` tràn).
- Luôn thử `[0, 1]`, số âm, `INT_MIN`/`INT_MAX`, và cặp `a ^ b` có bit `0` bằng nhau.

## 15. Biến thể của bài

1. **Single Number** (136): một số xuất hiện một lần, còn lại hai lần — XOR cả mảng.
2. **Single Number II** (137): còn lại xuất hiện **ba** lần — đếm bit modulo `3`.
3. **Missing Number** (268), **Find the Duplicate Number** (287): XOR / công thức tổng.
4. **Hai số lẻ lần** tổng quát: một số xuất hiện lẻ lần, hai số khác xuất hiện lẻ lần (cùng kỹ thuật).
5. **Hamming distance** hai số: `(a ^ b).bit_count()`.
6. **Hoán đổi hai số không dùng biến phụ**: `a ^= b; b ^= a; a ^= b`.
#### Code demo

In [22]:
def single_number_i(nums):
    return reduce(xor, nums)


def single_number_ii(nums):
    """Mọi số xuất hiện 3 lần trừ một số xuất hiện 1 lần (số có thể âm, dùng 32 bit)."""
    result = 0
    for bit in range(32):
        total = sum((x >> bit) & 1 for x in nums)
        if total % 3:
            result |= 1 << bit
    return result - (1 << 32) if result >= (1 << 31) else result            # đổi bù hai 32-bit về số có dấu


def swap_xor(a, b):
    a ^= b
    b ^= a
    a ^= b
    return a, b


print(single_number_i([4, 1, 2, 1, 2]))                  # 4
print(single_number_ii([2, 2, 3, 2]), single_number_ii([-5, 7, 7, 7]))      # 3 -5
print(swap_xor(3, 5))                                    # (5, 3)

4
3 -5
(5, 3)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hai vòng lặp | đếm từng phần tử bằng quét lại | `O(n²)` | `O(1)` | oracle, `n` nhỏ |
| 2. `Counter` | đếm tần suất | `O(n)` | `O(n)` | thực dụng, không ràng buộc bộ nhớ |
| 3. Sắp xếp | quét theo cặp | `O(n log n)` | `O(n)` | dữ liệu đã sắp hoặc cho phép sắp tại chỗ |
| 4. Set toggle | thêm/xóa khi gặp | `O(n)` | `O(n)` | viết nhanh |
| 5. XOR + phân nhóm | `xs`, `lb = xs & -xs`, XOR nhóm | `O(n)` | `O(1)` | **đúng yêu cầu đề** |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `6 ^ 6`; (b) `12 & -12`; (c) `reduce(xor, [2, 3, 2, 4, 4])`; (d) `Solution().singleNumber([2, 5, 2, 7])` (sắp xếp kết quả).

**Bài 2 (tính bằng tay).** Với `nums = [4, 1, 2, 1, 2, 6]`: tính `xs`, `lb`, hai nhóm và hai số đơn lẻ.

**Bài 3 (điền TODO).** Hoàn thiện `single_todo(nums)` theo Cách 5.
#### Code demo

In [23]:
def single_todo(nums):
    xs = 0
    # TODO 1: XOR mọi phần tử vào xs
    # TODO 2: lb = xs & -xs
    a = 0
    # TODO 3: với mỗi x có x & lb: a ^= x
    # TODO 4: trả [a, xs ^ a]
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples:
#     assert normalize(single_todo(nums)) == normalize(expected)

**Bài 4 (tự viết lại).** Viết `single_number_by_bit(nums)` bằng cách **thử từng vị trí bit** `b = 0..31` cho tới khi tìm được một bit mà `xs` có `1` (không dùng `x & -x`), rồi phân nhóm; kiểm tra với `oracle`.

**Bài 5 (biến thể / mini challenge).** Viết `lowest_set_bit_loop(x)` lấy bit `1` thấp nhất của `x` **bằng vòng lặp** (không dùng `x & -x`), rồi kiểm tra nó **bằng** `x & -x` trên cả số dương lẫn **số âm** (kể cả `-2^31`). Giải thích bằng lời vì sao `x & -x` vẫn đúng với số âm trong Python.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `a ^ a = 0`; `x & -x` giữ bit `1` thấp nhất; các cặp triệt tiêu. Bài 2: tìm hai số đơn lẻ rồi XOR chúng. Bài 4: tìm một vị trí `b` mà `(xs >> b) & 1 == 1`.
- **Mức 2 (vừa):** Bài 1: `0`, `4`, `3` (vì `2 ^ 3 ^ 2 ^ 4 ^ 4 = 3`), `[5, 7]`. Bài 2: hai số đơn lẻ là `4` và `6`; `xs = 4 ^ 6 = 2`, `lb = 2`; nhóm có bit `010`: `2`, `2`, `6` ⇒ `6`; nhóm kia: `4`, `1`, `1` ⇒ `4`. Bài 3: khung Cách 5.
- **Mức 3 (sâu):** Bài 4: `bit = 0; while not (xs >> bit) & 1: bit += 1; lb = 1 << bit`. Bài 5: `bit = 1; while not (x & bit): bit <<= 1`; với số âm Python dùng bù hai vô hạn (`-x = ~x + 1`), nên `x & -x` giữ đúng bit `1` thấp nhất; với `x = 0` thì kết quả `0` (không xảy ra vì `xs ≠ 0`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert 6 ^ 6 == 0 and 12 & -12 == 4
assert reduce(xor, [2, 3, 2, 4, 4]) == 3
assert sorted(Solution().singleNumber([2, 5, 2, 7])) == [5, 7]

# Bài 2
nums2 = [4, 1, 2, 1, 2, 6]
xs2 = reduce(xor, nums2)
assert xs2 == 4 ^ 6 == 2 and xs2 & -xs2 == 2
assert sorted(Solution().singleNumber(nums2)) == [4, 6]


# Bài 3
def single_todo_ans(nums):
    xs = 0
    for x in nums:
        xs ^= x
    lb = xs & -xs
    a = 0
    for x in nums:
        if x & lb:
            a ^= x
    return [a, xs ^ a]


for nums, expected in examples:
    assert normalize(single_todo_ans(nums)) == normalize(expected)


# Bài 4
def single_number_by_bit(nums):
    xs = 0
    for x in nums:
        xs ^= x
    bit = 0
    while not (xs >> bit) & 1:                       # tìm vị trí bit 1 thấp nhất bằng vòng lặp
        bit += 1
    lb = 1 << bit
    a = 0
    for x in nums:
        if x & lb:
            a ^= x
    return [a, xs ^ a]


rng = random.Random(14)
for _ in range(300):
    vals = rng.sample(range(-50, 50), rng.randint(0, 10) + 2)
    nums = make_input(vals[2:], vals[:2], rng)
    assert normalize(single_number_by_bit(nums)) == oracle(nums)

# Bài 5
def lowest_set_bit_loop(x):
    bit = 1
    while not (x & bit):
        bit <<= 1
    return bit


for x in (1, 2, 6, 12, 40, -1, -2, -3, -6, -8, -(2 ** 31), 2 ** 31 - 1):
    assert lowest_set_bit_loop(x) == x & -x
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (XOR cả mảng + phân nhóm theo bit thấp nhất)

```python
from functools import reduce
from operator import xor

class Solution:
    def singleNumber(self, nums: List[int]) -> List[int]:
        xs = reduce(xor, nums)          # = a ^ b
        lb = xs & -xs                   # một bit mà a và b khác nhau
        a = 0
        for x in nums:
            if x & lb:
                a ^= x
        return [a, xs ^ a]
```

**Checklist**

- [ ] XOR cả mảng cho `a ^ b` (khác `0`)?
- [ ] Bit phân biệt lấy từ **chính `xs`**: `xs & -xs` (không phải bit `0` cố định, không phải `xs & (xs - 1)`)?
- [ ] `a` khởi tạo bằng `0` và chỉ XOR các số có bit `lb` bật?
- [ ] Số kia là `xs ^ a`; trả về hai số (thứ tự bất kỳ)?
- [ ] Đã thử `[0, 1]`, số âm, `INT_MIN`/`INT_MAX`, và cặp có `a ^ b = 2` (bit `0` giống nhau)?
- [ ] Đã đối chiếu với `Counter` trên mảng ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def singleNumber(self, nums: List[int]) -> List[int]`?